In [2]:
import torch
from torch import nn,optim
from torch.utils.data import Dataset,DataLoader

In [3]:
# I didn't write this class

class XOR_Data(Dataset):

    # Constructor
    def __init__(self, N_s=100):
        self.x = torch.zeros((N_s, 2))
        self.y = torch.zeros((N_s, 1))
        for i in range(N_s // 4):
            self.x[i, :] = torch.Tensor([0.0, 0.0])
            self.y[i, 0] = torch.Tensor([0.0])

            self.x[i + N_s // 4, :] = torch.Tensor([0.0, 1.0])
            self.y[i + N_s // 4, 0] = torch.Tensor([1.0])

            self.x[i + N_s // 2, :] = torch.Tensor([1.0, 0.0])
            self.y[i + N_s // 2, 0] = torch.Tensor([1.0])

            self.x[i + 3 * N_s // 4, :] = torch.Tensor([1.0, 1.0])
            self.y[i + 3 * N_s // 4, 0] = torch.Tensor([0.0])

            self.x = self.x + 0.01 * torch.randn((N_s, 2))
        self.len = N_s

    # Getter
    def __getitem__(self, index):
        return self.x[index],self.y[index]

    # Get Length
    def __len__(self):
        return self.len

In [4]:
class Net(nn.Module):
    def __init__(self, d_in , h , d_out) -> None:
        super(Net, self).__init__()
        self.linear1 = nn.Linear(d_in , h)
        self.linear2 = nn.Linear(h, d_out)

    def forward(self, X):
        X = nn.functional.sigmoid(self.linear1(X))
        X = nn.functional.sigmoid(self.linear2(X))
        return X

In [5]:
def train_model(train_set, train_loader , optimizer , model , criterion , epoches):
    cost = []
    for epoch in range(epoches):
        total = 0
        for X_batch , y_batch in train_loader:
            optimizer.zero_grad()
            y_hat = model(X_batch)
            loss = criterion(y_hat , y_batch)
            loss.backward()
            optimizer.step()
            total += loss.item()
        cost.append(( total / len(train_set)))
    return cost

In [11]:
train_set = XOR_Data()
train_loader = DataLoader(train_set,batch_size=100)

criterion = nn.BCELoss()

In [12]:
model = Net(2,8,1)
optimizer = optim.Adam(model.parameters(),lr=0.01)

In [13]:
cost = train_model(train_set, train_loader,optimizer,model,criterion,5)
print(cost)

[0.00732530951499939, 0.0072466439008712765, 0.007177003622055054, 0.0071165281534194946, 0.007065246105194092]
